# Objective

Illustrate fine-tuning Mistral to follow a specific output format (for e.g., JSON).

# Setup

In [ ]:
# #@title Run this cell to setup Unsloth on Colab
# !pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# !pip install -q --no-deps xformers==0.0.28.post1 trl==0.11.1 peft==0.13.0 accelerate==0.34.2 bitsandbytes==0.44.1
# !pip install triton==3.0.0

In [ ]:
!pip install "trl<0.15.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.9/313.9 kB 11.8 MB/s eta 0:00:00


Run this cell to setup Unsloth on Colab

In [ ]:
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 MB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 868.6/868.6 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0

In [ ]:
#download datasets evaluate rouge_score and bert score
!pip install -q datasets==3.0.1 evaluate==0.4.3 bert_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 18.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.5.3 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 3.0.1 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [ ]:
import torch
import json

from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


# Model

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/mistral-7b-instruct-v0.2-bnb-4bit",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True
)

==((====))==  Unsloth 2026.5.5: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

In [ ]:
tokenizer.add_bos_token, tokenizer.add_eos_token

(False, False)

In [ ]:
EOS_TOKEN = tokenizer.eos_token

# Data

In [ ]:
dataset = load_dataset("pgurazada1/entities-laptop")

README.md:   0%|          | 0.00/154 [00:00<?, ?B/s]

entities-train.json: 0.00B [00:00, ?B/s]

entities-validation.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/39 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/20 [00:00<?, ? examples/s]

In [ ]:
training_dataset = dataset['train']
validation_dataset = dataset['validation']

In [ ]:
training_dataset[0]

{'entities': {'battery_life': 'impressive, lasting a solid 10 hours',
  'design': 'sleek',
  'keyboard': 'backlit',
  'performance': 'handles multitasking effortlessly',
  'sturdiness': 'aluminum chassis',
  'trackpad': 'responsive'},
 'review': "The laptop's battery life is impressive, lasting a solid 10 hours on a single charge. The sleek design and backlit keyboard enhance the overall experience. Performance-wise, it handles multitasking effortlessly. The aluminum chassis adds sturdiness, and the trackpad is responsive."}

In [ ]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

In [ ]:
def prompt_formatter(example, prompt_template):
    instruction="Extract entities in the input review in a JSON format."
    review=example["review"]
    entities=example["entities"]

    formatted_prompt = prompt_template.format(instruction, review, entities) + EOS_TOKEN

    return {'formatted_prompt': formatted_prompt}

In [ ]:
formatted_training_dataset = training_dataset.map(
    prompt_formatter,
    fn_kwargs={'prompt_template': alpaca_prompt}
)

Map:   0%|          | 0/39 [00:00<?, ? examples/s]

In [ ]:
formatted_validation_dataset = validation_dataset.map(
    prompt_formatter,
    fn_kwargs={'prompt_template': alpaca_prompt}
)

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [ ]:
formatted_training_dataset[0]

{'entities': {'battery_life': 'impressive, lasting a solid 10 hours',
  'design': 'sleek',
  'keyboard': 'backlit',
  'performance': 'handles multitasking effortlessly',
  'sturdiness': 'aluminum chassis',
  'trackpad': 'responsive'},
 'review': "The laptop's battery life is impressive, lasting a solid 10 hours on a single charge. The sleek design and backlit keyboard enhance the overall experience. Performance-wise, it handles multitasking effortlessly. The aluminum chassis adds sturdiness, and the trackpad is responsive.",
 'formatted_prompt': "Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nExtract entities in the input review in a JSON format.\n\n### Input:\nThe laptop's battery life is impressive, lasting a solid 10 hours on a single charge. The sleek design and backlit keyboard enhance the overall experience. Performance-wise, it handles multitasking 

# Fine-tuning

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=4,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing=True,
    random_state=42,
    loftq_config=None
)

Unsloth 2026.5.5 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Notice how $r = \alpha/4$.

In [ ]:
def formatting_func(example):
    # example["formatted_prompt"] can be:
    #  - a single string
    #  - a list of strings (in batched mode)

    text = example["formatted_prompt"]

    # if it is already a list -> return as-is
    if isinstance(text, list):
        return text

    # if it is a single string -> wrap inside a list
    return [text]

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_training_dataset,
    eval_dataset=formatted_validation_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    formatting_func=formatting_func,
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=20,
        eval_strategy="epoch",
        save_strategy='epoch',
        metric_for_best_model="eval_loss",
        load_best_model_at_end=True,
        greater_is_better=False,
        learning_rate=5e-5,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="paged_adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        report_to="none",
        output_dir="outputs"
    )
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/39 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/20 [00:00<?, ? examples/s]

In [ ]:
training_history = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 39 | Num Epochs = 20 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 10,485,760 of 7,252,217,856 (0.14% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss
1,1.519043,1.237874
2,0.809202,0.706613
3,0.491266,0.364441
4,0.320125,0.236132
5,0.281961,0.187654
6,0.210344,0.166263
7,0.208003,0.153089
8,0.159445,0.141701
9,0.161983,0.136803
10,0.129904,0.125942


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-5.
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`

# Inference

In [ ]:
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32000, 4096, padding_idx=0)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=4, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=4, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [ ]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32000, 4096, padding_idx=0)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=4, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=4, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [ ]:
test_review = """
This laptop impresses with its remarkable battery life, lasting well beyond the competition. The sleek design adds a touch of elegance, complemented by a comfortable keyboard that enhances productivity. Performance is stellar, seamlessly handling demanding tasks. Sturdiness is evident in its durable build, ensuring longevity. The trackpad is responsive and precise, contributing to an overall delightful user experience. In summary, this laptop excels across the board, making it a top choice for those seeking a reliable and high-performing device.
"""

In [ ]:
instruction = "Extract entities in the input review in a JSON format."

In [ ]:
inputs = tokenizer(
[
    alpaca_prompt.format(
        instruction,
        test_review,
        "", # leave output blank for generation
    )
], return_tensors="pt").to("cuda")

In [ ]:
outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    use_cache=True,
    pad_token_id=tokenizer.eos_token_id
)

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

In [ ]:
print(
    tokenizer.decode(
        outputs[0][inputs.input_ids.shape[-1]:],
        skip_special_tokens=True,
        cleanup_tokenization_spaces=True
    )
)

{'battery_life': 'remarkable, lasting well beyond the competition', 'design': 'sleek, adds a touch of elegance', 'keyboard': 'comfortable, enhances productivity', 'performance': 'stellar, seamlessly handling demanding tasks', 'sturdiness': 'durable build, ensures longevity', 'trackpad': 'responsive and precise', 'user_experience': 'overall delightful'}


# Evaluation

In [ ]:
examples = [
    "The Dell XPS 13 impresses with its sleek design and remarkable battery life. " \
    "The backlit keyboard offers a comfortable typing experience, while the powerful performance ensures seamless multitasking. " \
    "Sturdiness is top-notch, but the trackpad could be more responsive.",
    "Apple's MacBook Air M2 is a design masterpiece, ultra-thin and lightweight. "
    "The keyboard is a joy to type on, but its standout feature is the incredible battery life. "\
    "Performance-wise, it's a powerhouse, though the sturdiness could be enhanced.",
    "The HP Spectre x360 blends elegance with versatility. "\
    "The keyboard is tactile, and the battery life is commendable. "\
    "Performance-wise, it's a workhorse, but the sturdiness feels slightly compromised. "\
    "The trackpad, however, is responsive and intuitive.",
    "Lenovo's ThinkPad X1 Carbon exudes durability with its robust design. "\
    "The keyboard is superb for long typing sessions. "\
    "Although the battery life falls short of some competitors, the performance is stellar. "\
    "The trackpad is precise, but the design lacks a modern flair.",
    "The Asus ROG Zephyrus G14 is a powerhouse for gaming and productivity. "\
    "While the design is gaming-centric, the keyboard is comfortable, and the performance is exceptional. "\
    "Battery life is average, but the sturdiness is noteworthy. The trackpad could use some improvement."
]

In [ ]:
for example in examples:
  inputs = tokenizer(
      [
          alpaca_prompt.format(
              instruction,
              example,
              "", # leave output blank for generation
          )
      ], return_tensors="pt").to("cuda")

  outputs = model.generate(
    **inputs,
    max_new_tokens=128,
    temperature=0,
    use_cache=True,
    pad_token_id=tokenizer.eos_token_id
  )

  output_json = tokenizer.decode(
      outputs[0][inputs.input_ids.shape[-1]:],
      skip_special_tokens=True,
      cleanup_tokenization_spaces=True
  )

  print(example)
  print(output_json.replace("'", '"'))
  print("***\n")

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The Dell XPS 13 impresses with its sleek design and remarkable battery life. The backlit keyboard offers a comfortable typing experience, while the powerful performance ensures seamless multitasking. Sturdiness is top-notch, but the trackpad could be more responsive.
{"design": "sleek", "battery_life": "remarkable", "keyboard": "backlit, comfortable", "performance": "seamless multitasking", "sturdiness": "top-notch", "trackpad": "could be more responsive"}
***



Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Apple's MacBook Air M2 is a design masterpiece, ultra-thin and lightweight. The keyboard is a joy to type on, but its standout feature is the incredible battery life. Performance-wise, it's a powerhouse, though the sturdiness could be enhanced.
{"design": "design masterpiece, ultra-thin and lightweight", "keyboard": "keyboard is a joy to type on", "battery_life": "incredible battery life", "performance": "powerhouse", "sturdiness": "could be enhanced"}
***



/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The HP Spectre x360 blends elegance with versatility. The keyboard is tactile, and the battery life is commendable. Performance-wise, it's a workhorse, but the sturdiness feels slightly compromised. The trackpad, however, is responsive and intuitive.
{"keyboard": "tactile", "battery_life": "commendable", "performance": "workhorse", "sturdiness": "slightly compromised", "trackpad": "responsive and intuitive"}
***



Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Lenovo's ThinkPad X1 Carbon exudes durability with its robust design. The keyboard is superb for long typing sessions. Although the battery life falls short of some competitors, the performance is stellar. The trackpad is precise, but the design lacks a modern flair.
{"durability": "exudes durability with its robust design", "keyboard": "superb for long typing sessions", "battery_life": "falls short of some competitors", "performance": "stellar", "trackpad": "precise", "design": "lacks a modern flair"}
***

The Asus ROG Zephyrus G14 is a powerhouse for gaming and productivity. While the design is gaming-centric, the keyboard is comfortable, and the performance is exceptional. Battery life is average, but the sturdiness is noteworthy. The trackpad could use some improvement.
{"design": "gaming-centric", "keyboard": "comfortable", "performance": "exceptional", "battery_life": "average", "sturdiness": "noteworthy", "trackpad": "could use some improvement"}
***



In [ ]:
# Export the model locally as a 5-bit GGUF
model.save_pretrained_gguf(
    "mistral_7b_q5_gguf",
    tokenizer,
    quantization_method = "q5_k_m"
)

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/722 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in mistral_7b_q5_gguf/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in mistral_7b_q5_gguf.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...


Unsloth: Preparing safetensor model files:  67%|██████▋   | 2/3 [05:15<02:49, 169.01s/it]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 3/3 [09:29<00:00, 189.87s/it]

Unsloth: Merging weights into 16bit: 100%|██████████| 3/3 [07:06<00:00, 142.17s/it]


Unsloth: Merge process complete. Saved to `/content/mistral_7b_q5_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q5_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['mistral_7b_q5_gguf_gguf/mistral-7b-instruct-v0.2.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q5_k_m. This might take 10 minutes...
Uns

{'save_directory': 'mistral_7b_q5_gguf',
 'gguf_directory': 'mistral_7b_q5_gguf_gguf',
 'gguf_files': ['mistral_7b_q5_gguf_gguf/mistral-7b-instruct-v0.2.Q5_K_M.gguf'],
 'modelfile_location': 'mistral_7b_q5_gguf_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp -r /content/mistral_7b_q5_gguf_gguf /content/drive/MyDrive/